<a href="https://colab.research.google.com/github/TheoSarpey/lab-guided-tutorial-transformers/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab | Transformers

---

**Note: for GPU acceleration, run this notebook in [Google Colab](https://colab.research.google.com/).**



### Section structure

1. The open-source ecosystem: increasing accessibility to machine learning (ML) software and hardware
2. Some simple code demonstrations
3. Q&A

## 1. Ease-of-use: Using Transformers in 3 lines of code


**Overview of different tasks that can be automated with ML**
* Key ingredients: (1) a model trained on a specific task; (2) input data (e.g. texts or images); (3) output produced by the model.
* Transformers are currently the most popular type of deep learning algorithm. Most tasks below are solved with Transformers. There might be other types of algorithms coming up in the medium term.



**Install the Transformers library & dependencies**

In [1]:
!pip install transformers  # The Transformers library from Hugging Face
!pip install sentencepiece
!pip install wikipedia
!pip install accelerate
!pip install tf-keras
!pip install torch

# NOTE: you might need to restart you jupyter kernel after installing the libraries

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11757 sha256=bdafc249af4c4d8a6075f9e5fe35571194712ab4f4cd41818cfaaf23b833ac7b
  Stored in directory: /root/.cache/pip/wheels/79/1d/c8/b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia


**The Hugging Face Pipeline**
* Makes automation of many NLP tasks possible in 3 lines of code
* Detailed documentation is available [here](https://huggingface.co/transformers/main_classes/pipelines.html)

In [2]:
from transformers import pipeline
import pandas as pd
import numpy as np
from pprint import pprint

Note : You might need more libraries or updates to run the cells below, if that is the case, follow the error messages and pip install accordingly. Chat gpt can help you if given the error messages.

### 2.1 Many models tailored to specific tasks


#### 2.1.1 Text classification

Let's select a popular text classification model in the [HF model hub](https://huggingface.co/models?pipeline_tag=text-classification&sort=downloads).

Here we chose "cardiffnlp/twitter-roberta-base-irony".

We will classify text into ironic or non ironic.

In [5]:
pipeline_classification = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-irony")  # cardiffnlp/twitter-roberta-base-irony, SamLowe/roberta-base-go_emotions

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Now that we have the model we can pass it a string and have it give us a classification.

Feel free to experiment with different sentences by changing the contents of the variable text

In [6]:
text = "Well that workshop was totally worth my time..."  # "Well that workshop was totally worth my time..."  "This smells weird, I'm not sure if I should eat this ... Yikes, it tasted like old socks!"
output = pipeline_classification(text, top_k=10)
print(output)

[{'label': 'irony', 'score': 0.942438542842865}, {'label': 'non_irony', 'score': 0.05756139010190964}]


Let's make the output a little cleaner

In [7]:
# make output a bit cleaner
df_output = pd.DataFrame(output)
print(df_output)

       label     score
0      irony  0.942439
1  non_irony  0.057561


As you can see, in a few lines of code and by leveraging an existing model we can classify text as ironic or non ironic. Now you have one more tool in your machine learning toolbox.

Remember that : 'when you only have a hammer everything is a nail'. But if we want to build a house (perform machine learning the right way), we need to use the right tool for the right job.

#### 2.1.2 Machine Translation

* Open source machine translation (MT) models enable you to translate between many different languages without Google Translate.
* [University of Helsinki](https://huggingface.co/Helsinki-NLP) uploaded models for more than 1000 language pairs to the Hugging Face hub
* [Facebook AI](https://huggingface.co/models?search=facebook+m2m) open-sourced several multi-lingual models
* The [EasyNMT library](https://github.com/UKPLab/EasyNMT), provides an easy wrapper for all these models
* Most machine translation models translate between two languages in one direction (e.g. German to English, but not English to German), some can translate in multiple directions.


In [9]:
# We load the model and tokenizer directly for Seq2Seq tasks, since the general
# “translation” pipeline has been removed from Transformers v5.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/m2m100_418M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# We'll create a function to handle the translation
def pipeline_translate(text, src_lang="de", tgt_lang="en"):
    tokenizer.src_lang = src_lang
    encoded_input = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(**encoded_input, forced_bos_token_id=tokenizer.get_lang_id(tgt_lang))
    translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return [{"translation_text": translated_text}]

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 2.42MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.94GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

M2M100 is a multilingual encoder-decoder (seq-to-seq) model trained for Many-to-Many multilingual translation.

The model that can directly translate between the 9,900 directions of 100 languages.

Here we specify to translate from German 'de' to English 'en'

In [10]:
text = "Ich bin ein Fisch"
pipeline_translate(text, src_lang="de", tgt_lang="en")

[{'translation_text': 'I am a fish'}]

Let's do the same but with and entire wikipedia page in german.

In [12]:
# download any text from wikipedia, via  https://pypi.org/project/wikipedia/
import wikipedia

# We set a user-agent to prevent Wikipedia from blocking the request
wikipedia.set_user_agent("MyColabTranslationBot/1.0 (contact: test@example.com)")
wikipedia.set_lang("de")

try:
    text = wikipedia.summary("Donald Trump").replace('\n', ' ')[:318]
    print(f"Original text:\n{text}\n")

    # translate the text from wikipedia
    text_translated = pipeline_translate(text, src_lang="de", tgt_lang="en")
    print(f"Translated text:\n{text_translated[0]['translation_text']}")
except Exception as e:
    print(f"Ein Fehler ist aufgetreten: {e}")

Original text:
Donald John Trump [ˈdɑn.əld dʒɑn tɹɐmp] (* 14. Juni 1946 in Queens, New York City) ist ein US-amerikanischer Politiker der Republikanischen Partei. Er war von 2017 bis 2021 der 45. und ist seit dem 20. Januar 2025 der 47. Präsident der Vereinigten Staaten. Außerdem ist er Unternehmer und ehemaliger Showmaster. Er gil

Translated text:
Donald John Trump [ˈdɑn.əld dʒɑn tɔmp] (* 14 June 1946 in Queens, New York City) is an American politician of the Republican Party. he was from 2017 to 2021 the 45th and is from 20 January 2025 the 47th President of the United States.


#### 2.1.3 Text Summarization

In [14]:
# Since the general “summarization” pipeline was removed in Transformers v5, we
# load the model and the tokenizer directly using the Auto classes.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer_summarize = AutoTokenizer.from_pretrained(model_name)
model_summarize = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# A helper function for summarization that is compatible with the existing code
def pipeline_summarize(text, min_length=5, max_length=30):
    inputs = tokenizer_summarize(text, max_length=1024, truncation=True, return_tensors="pt")
    summary_ids = model_summarize.generate(inputs["input_ids"], min_length=min_length, max_length=max_length, num_beams=4, early_stopping=True)
    summary_text = tokenizer_summarize.decode(summary_ids[0], skip_special_tokens=True)
    return [{"summary_text": summary_text}]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

In [15]:
# download any long text from wikipedia, via  https://pypi.org/project/wikipedia/
import wikipedia
wikipedia.set_lang("en")

text_long = wikipedia.summary("Donald Trump").replace('\n', ' ')
print(f"Original text:\n{text_long}\n")

# translate the text from wikipedia
text_summarized = pipeline_summarize(text_long, min_length=5, max_length=30)
print(f"Summarized text:\n{text_summarized[0]['summary_text']}")

Original text:
Donald John Trump (born June 14, 1946) is an American politician, media personality, and businessman who is the 47th president of the United States. A member of the Republican Party, he served as the 45th president from 2017 to 2021. Born into a wealthy New York City family, Trump graduated from the University of Pennsylvania in 1968 with a bachelor's degree in economics. He became the president of his family's real estate business in 1971, renamed it the Trump Organization, and began acquiring and building skyscrapers, hotels, casinos, and golf courses. He launched side ventures, many licensing the Trump name, and filed for six business bankruptcies in the 1990s and 2000s. From 2004 to 2015, he hosted the reality television show The Apprentice, bolstering his reputation as a wealthy tycoon. Presenting himself as a political outsider, Trump won the 2016 presidential election against Democratic Party nominee Hillary Clinton. During his first presidency, Trump imposed a tr

#### 2.1.4 Named Entity Recognition

NER is a task that involves identifying and classifying specific entities in text into predefined categories, such as names of people, organizations, locations, dates, and more.

For example, in the sentence "Apple Inc. was founded by Steve Jobs in California," NER would recognize "Apple Inc." as an organization, "Steve Jobs" as a person, and "California" as a location.

In [16]:
pipeline_ner = pipeline("token-classification", model="dslim/bert-base-NER-uncased", aggregation_strategy="simple")

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER-uncased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [17]:
import wikipedia
wikipedia.set_lang("en")

text_long = wikipedia.summary("Donald Trump").replace('\n', ' ')

output = pipeline_ner(text_long)

pd.DataFrame(output)

,entity_group,score,word,start,end
0,PER,0.985104,donald john trump,0,17
1,MISC,0.984194,american,45,53
2,LOC,0.986981,united states,134,147
3,MISC,0.789240,republican,165,175
4,ORG,0.722176,party,176,181
5,LOC,0.891513,new york city,254,267
6,PER,0.977508,trump,276,281
7,ORG,0.668195,university of pennsylvania,301,327
8,ORG,0.849596,trump organization,460,478
9,PER,0.920151,trump,607,612


### 2.2. Universal models

The models mentioned above are designed to excel at a single specific task on a particular dataset. The key advantage of these models is their high performance and accuracy on that specific task and dataset.

However, in real-world applications, the problems you'll face often require solving slightly different tasks, possibly with varied category definitions or applied to different types of texts.

Universal models can help address this challenge. Although they also focus on one task, the task is general or universal enough that many other tasks can be reformulated into it. Two examples of universal tasks are:

- Natural Language Inference (NLI): A task that can effectively solve a wide range of classification tasks by determining whether a given premise supports, contradicts, or is neutral with respect to a hypothesis.

- Token Generation: An even more universal task that can be applied to solve virtually any text-related task, including translation, summarization, and text completion.

These universal tasks enable the models to be versatile and adaptable to various problems beyond the specific ones they were initially trained on.

#### Zero-shot classification


Zero-shot classification is a technique where a model can categorize data into classes it has never seen before.

Instead of relying on labeled examples for each class, the model understands the relationship between the input and the class descriptions, allowing it to make accurate predictions without needing specific training on those classes.

In [18]:
pipeline_zeroshot_classification = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Here we will give the model a list of classes ('payment issues', 'travel advice', 'bug report') for it to classify our string.

In [19]:
text = "Customer: I have not received my reimbursement yet. What the hell is going on?"
classes = ['payment issues', 'travel advice', 'bug report']  # "account opening", "customer complaint"

#text = "I do not think the government is trustworthy anymore. We need to mobilize and resist!"
#classes = ["civil disobedience", "praise of the government", "travel advice"]  # "collective action"

output = pipeline_zeroshot_classification(text, classes, multi_label=True)

pd.DataFrame(data=[output["labels"], output["scores"]], index=["class", "probability"]).T


,class,probability
0,payment issues,0.99111
1,bug report,0.076667
2,travel advice,0.018878


## Exercise

Now it is your turn to go to the hugging face library https://huggingface.co/models?pipeline_tag=text-classification&sort=downloads

(you can select on the left menu of the website the type of NLP tasks you want models to perform)

- Find an NLP model that we have not used previously.

- Get some data from wikipedia or elsewhere.

- Perform inference with the model and print the result!

- Comment your code along the way, describe what your model does and what your end goal is from input to output.

Have fun!

In [ ]:
# Your code here :

# Lab | Transformers: practical NLP with Hugging Face

**Goal:** understand how pretrained Transformer models turn raw text into useful predictions with minimal code, and learn how to inspect those predictions critically.

By the end of this lab, I would have tried:

1. Text classification (irony detection)
2. Multilingual machine translation
3. Abstractive text summarization
4. Named Entity Recognition (NER)
5. Zero-shot classification with user-defined labels
6. An independent model experiment using emotion classification

> **Runtime note:** these examples download pretrained models the first time they run. Some models are large, so a stable internet connection and several minutes may be required. A GPU (for example, Google Colab with a GPU runtime) can speed up some tasks, but is not essential for every demonstration.

## Important caveats

- Treat model outputs as predictions, not ground truth. Confidence scores are not guarantees of correctness.
- If a download or package installation fails, resolve that environment issue and rerun the affected cell.

## 1. Setup: the Hugging Face ecosystem

Hugging Face's `transformers` library provides a consistent **pipeline** interface for common machine-learning tasks. A pipeline generally handles preprocessing, model inference, and postprocessing.

It is vital to run a few package installation commands. Since I used a managed environment, i.e., Google Colab, I only installed what I need.

In [20]:
# Run this cell only if the packages are not already installed in your environment.
%pip install -q transformers sentencepiece wikipedia accelerate torch pandas numpy

### Imports

We use pandas to display model outputs in tables and NumPy for general numerical work. `pprint` is useful for inspecting nested Python objects.

In [21]:
from transformers import pipeline
import pandas as pd
import numpy as np
from pprint import pprint

pd.set_option("display.max_colwidth", 120)

## 2. Task-specific pretrained models

A task-specific model is trained or fine-tuned for a particular job. The quality of its output depends on the model, its training data, the input language, and whether your use case resembles the task it learned.

### 2.1 Text classification: irony detection

The model `cardiffnlp/twitter-roberta-base-irony` predicts whether a short text is ironic. Because it was trained on Twitter-style text, its behavior may be less reliable on formal writing or other domains.

In [22]:
# Load a pretrained irony classifier. The first run downloads the model files.
pipeline_classification = pipeline(
    task="text-classification",
    model="cardiffnlp/twitter-roberta-base-irony",
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Try a sentence with a potentially sarcastic tone. The meaning of irony depends heavily on context, so inspect the score as model confidence rather than a calibrated probability of truth.

In [23]:
text_irony = "Well that workshop was totally worth my time..."

# Return both label scores where the model supports them.
irony_output = pipeline_classification(text_irony, top_k=None)
irony_df = pd.DataFrame(irony_output)
irony_df.sort_values("score", ascending=False).reset_index(drop=True)

,label,score
0,irony,0.942439
1,non_irony,0.057561


### 2.2 Machine translation

The multilingual `facebook/m2m100_418M` model can translate between many language pairs. Unlike a single-direction model, M2M100 lets us specify the source and target language codes at inference time.

Here, `de` means German and `en` means English. Translation may preserve the general meaning while losing nuance, idioms, or ambiguity.

In [25]:
# Since the general “translation” pipeline was removed in Transformers v5, we load
# the model and the tokenizer directly through the Auto classes.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/m2m100_418M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# A helper function that handles the translation in the same way as the previous pipeline
def pipeline_translate(text, src_lang="de", tgt_lang="en"):
    tokenizer.src_lang = src_lang
    encoded_input = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(**encoded_input, forced_bos_token_id=tokenizer.get_lang_id(tgt_lang))
    translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return [{"translation_text": translated_text}]

text_german = "Ich bin ein Fisch."
translation = pipeline_translate(text_german, src_lang="de", tgt_lang="en")
print("German:", text_german)
print("English:", translation[0]["translation_text"])

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

German: Ich bin ein Fisch.
English: I am a fish.


#### Translate a longer real-world passage

The original exercise uses a Wikipedia summary. Network access and Wikipedia page availability can vary, so this cell includes a short fallback passage. Keep the input reasonably short for a quick demonstration.

In [26]:
import wikipedia

try:
    wikipedia.set_lang("de")
    source_text = wikipedia.summary("Deutschland", sentences=3, auto_suggest=False)
except Exception as exc:
    print(f"Wikipedia retrieval was unavailable ({type(exc).__name__}); using a fallback passage.")
    source_text = (
        "Deutschland liegt in Mitteleuropa. Die Hauptstadt ist Berlin. "
        "Das Land ist Mitglied der Europäischen Union."
    )

# Limit input length to keep this demonstration manageable.
source_text = " ".join(source_text.split())[:700]
translated = pipeline_translate(source_text, src_lang="de", tgt_lang="en")
print("SOURCE (German):\n", source_text, "\n")
print("TRANSLATION (English):\n", translated[0]["translation_text"])

SOURCE (German):
 Deutschland (; Vollform des Staatsnamens: Bundesrepublik Deutschland) ist ein Bundesstaat in Mitteleuropa. Es besteht aus 16 Ländern und ist als freiheitlich-demokratischer und sozialer Rechtsstaat verfasst. Die 1949 gegründete Bundesrepublik Deutschland stellt die jüngste Ausprägung des 1871 erstmals gegründeten deutschen Nationalstaats dar. 

TRANSLATION (English):
 Germany (; full form of the state name: Federal Republic of Germany) is a state in Central Europe. it consists of 16 countries and is written as a freedom-democratic and social law state. the 1949 founded Federal Republic of Germany represents the latest expression of the first German national state founded in 1871.


### 2.3 Text summarization

Summarization generates a shorter version of a longer passage. The `sshleifer/distilbart-cnn-12-6` model is an abstractive summarizer: it can rephrase content rather than only selecting existing sentences.

**Important:** generated summaries can omit key facts or introduce unsupported details. Always compare a summary with its source when accuracy matters.

In [28]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load the model and tokenizer directly, since the summarization pipeline has been removed
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer_summarize = AutoTokenizer.from_pretrained(model_name)
model_summarize = AutoModelForSeq2SeqLM.from_pretrained(model_name)

sample_article = (
    "Renewable energy comes from sources that are naturally replenished, such as sunlight, wind, "
    "and flowing water. Solar panels convert sunlight into electricity, while wind turbines use "
    "moving air to turn a generator. These technologies can reduce greenhouse-gas emissions, "
    "although their output varies with weather and time of day. Energy storage, transmission "
    "networks, and flexible demand can help balance supply and demand. Costs have fallen for "
    "many renewable technologies, but planning, land use, and materials remain important considerations."
)

# Tokenization and Generation of the Summary
inputs = tokenizer_summarize(sample_article, max_length=1024, truncation=True, return_tensors="pt")
summary_ids = model_summarize.generate(
    inputs["input_ids"],
    min_length=15,
    max_length=60,
    num_beams=4,
    early_stopping=True
)
summary_text = tokenizer_summarize.decode(summary_ids[0], skip_special_tokens=True)

print("ORIGINAL:\n", sample_article, "\n")
print("SUMMARY:\n", summary_text)

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

ORIGINAL:
 Renewable energy comes from sources that are naturally replenished, such as sunlight, wind, and flowing water. Solar panels convert sunlight into electricity, while wind turbines use moving air to turn a generator. These technologies can reduce greenhouse-gas emissions, although their output varies with weather and time of day. Energy storage, transmission networks, and flexible demand can help balance supply and demand. Costs have fallen for many renewable technologies, but planning, land use, and materials remain important considerations. 

SUMMARY:
  Renewable energy comes from sources that are naturally replenished, such as sunlight, wind and flowing water . Costs have fallen for many renewable technologies, but planning, land use, and materials remain important considerations .


### 2.4 Named Entity Recognition (NER)

NER identifies spans of text that refer to entities such as people (`PER`), organizations (`ORG`), and locations (`LOC`). The model below uses token classification and combines word-piece predictions into entity spans using `aggregation_strategy="simple"`.

In [29]:
pipeline_ner = pipeline(
    task="token-classification",
    model="dslim/bert-base-NER-uncased",
    aggregation_strategy="simple",
)

ner_text = "Ada Lovelace worked with Charles Babbage in London on an early computing machine."
ner_output = pipeline_ner(ner_text)
ner_df = pd.DataFrame(ner_output)
print("Text:", ner_text)
ner_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER-uncased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text: Ada Lovelace worked with Charles Babbage in London on an early computing machine.


,entity_group,score,word,start,end
0,PER,0.997864,ada lovelace,0,12
1,PER,0.996789,charles babbage,25,40
2,LOC,0.998267,london,44,50


**Interpretation:** inspect `entity_group`, `word`, and `score`. Entity labels are model-specific, and a high score does not guarantee that a span is correctly classified. NER systems may also miss entities or split/merge spans incorrectly.

## 3. More general task formulations: zero-shot classification

A zero-shot classifier can compare text with candidate labels that were not provided as task-specific training examples. This is useful when you need a quick baseline or your label set changes often. It does **not** mean the model has no prior training: the model was trained on other data and objectives.

With `multi_label=False`, the scores are normalized across candidate labels as competing alternatives. With `multi_label=True`, each label is scored independently, so scores need not sum to one. Choose the setting based on whether several labels may apply at once.

In [30]:
pipeline_zeroshot_classification = pipeline(
    task="zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

### Try custom labels

The candidate labels should be clear, distinct, and at a similar level of specificity. Here, one label is selected as the best match among three possible support topics.

In [33]:
customer_message = "I have not received my reimbursement yet. What is happening?"
candidate_labels = ["payment issues", "travel advice", "bug report"]

zero_shot_output = pipeline_zeroshot_classification(
    customer_message,
    candidate_labels=candidate_labels,
    multi_label=False,
)

zero_shot_df = pd.DataFrame({
    "label": zero_shot_output["labels"],
    "score": zero_shot_output["scores"],
})
zero_shot_df

,label,score
0,payment issues,0.949348
1,bug report,0.030044
2,travel advice,0.020608


**Reflection:** rewrite the candidate labels in different words and run the cell again. Do the rankings change? This is a reminder that zero-shot predictions depend on label wording and should be validated against labelled examples before deployment.

## 4. Independent exercise: emotion classification

Now explore a model not used in the demonstrations above: `j-hartmann/emotion-english-distilroberta-base`. It predicts emotion categories such as anger, disgust, fear, joy, neutral, sadness, and surprise from English text.

**Goal:** take several short messages as input, rank the predicted emotion labels, and compare the predictions with your own interpretation. This is a demonstration, not a reliable assessment of a person's internal emotional state.

In [32]:
# Load a pretrained emotion classifier (first run downloads its model files).
pipeline_emotion = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
)

messages = [
    "I got the job offer I was hoping for!",
    "I have been waiting for hours and nobody has helped me.",
    "The package arrived yesterday.",
    "I am worried about the results of tomorrow's exam.",
]

# Keep the top three labels for each message so the alternatives are visible.
emotion_rows = []
for message in messages:
    predictions = pipeline_emotion(message)
    # Since the pipeline returns a nested list [[{...}]] when `top_k=None',
    # we access the first element, `predictions[0]`, to flatten the list.
    top_predictions = sorted(predictions[0], key=lambda item: item["score"], reverse=True)[:3]
    for rank, prediction in enumerate(top_predictions, start=1):
        emotion_rows.append({
            "message": message,
            "rank": rank,
            "emotion": prediction["label"],
            "score": prediction["score"],
        })

emotion_results = pd.DataFrame(emotion_rows)
emotion_results

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,message,rank,emotion,score
0,I got the job offer I was hoping for!,1,joy,0.875280
1,I got the job offer I was hoping for!,2,surprise,0.076816
2,I got the job offer I was hoping for!,3,neutral,0.019515
3,I have been waiting for hours and nobody has helped me.,1,sadness,0.826277
4,I have been waiting for hours and nobody has helped me.,2,anger,0.100508
5,I have been waiting for hours and nobody has helped me.,3,neutral,0.028033
6,The package arrived yesterday.,1,joy,0.470253
7,The package arrived yesterday.,2,neutral,0.322968
8,The package arrived yesterday.,3,surprise,0.181882
9,I am worried about the results of tomorrow's exam.,1,fear,0.989695


## Key Takeaway
**Conclusion:** pretrained pipelines make NLP experimentation accessible, but choosing the right task/model, validating outputs, protecting privacy, and understanding failure modes remain essential.